# Credit Scoring Model Using Random Forest

## Section 1: Project Introduction

### Objective
Build a machine-learning model to estimate credit risk from the financial and loan-application fields available in the supplied dataset.

### Introduction
Credit scoring uses historical customer information to estimate whether a borrower is likely to repay a loan. The source `loan_status` column records default (`1`) versus no default (`0`). For clarity, this notebook reverses the labels into `creditworthy`: **1 = Good Credit / Low Default Risk** and **0 = Bad Credit / High Default Risk**.

### Random Forest
A Random Forest combines many decision trees. Their combined prediction can capture non-linear relationships and is less dependent on any single tree. The model's score for the bad-credit class is an estimate from this model, not a calibrated real-world probability.

## Section 2: Import Libraries

Import the data, plotting, preprocessing, modeling, and evaluation tools used in this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
    RocCurveDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## Section 3: Dataset Loading

Load the CSV located beside this notebook. Review its first records, dimensions, column names, and data types.

In [ ]:
df = pd.read_csv("credit_risk_dataset.csv")

display(df.head())
print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())
df.info()

## Section 4: Exploratory Data Analysis

Check data quality and summarize the data before modeling. Visualizations focus on target balance, numeric distributions, and numeric correlations.

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
print("Statistical summary:")
display(df.describe(include="all").T)

print("Source target counts (1 = default, 0 = no default):")
display(df["loan_status"].value_counts().sort_index().to_frame("count"))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="loan_status")
plt.title("Loan Status Distribution (1 = Default)")
plt.xlabel("loan_status")
plt.ylabel("Number of customers")
plt.show()

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
df[numeric_columns].hist(figsize=(15, 11), bins=30, edgecolor="black")
plt.suptitle("Numerical Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 8))
sns.heatmap(df[numeric_columns].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap for Numerical Columns")
plt.tight_layout()
plt.show()

## Section 5: Data Preprocessing

Remove exact duplicate rows, then split the data into 80% training and 20% testing with stratification. The target is reversed consistently: original default `1` maps to bad credit `0`, and original no-default `0` maps to good credit `1`.

The numeric and categorical imputers and one-hot encoder are placed in a scikit-learn pipeline, so they learn from training data only. No scaling is needed for a Random Forest.

In [ ]:
df = df.drop_duplicates().copy()

# 1 = good / low risk; 0 = bad / high risk.
y = (1 - df["loan_status"].astype(int)).rename("creditworthy")

# These columns exist in the CSV. Exclude loan_grade and loan_int_rate because
# they are underwriting-stage fields, and exclude loan_percent_income because
# it duplicates the engineered loan-to-income ratio.
numeric_features = [
    "person_age", "person_income", "person_emp_length", "loan_amnt",
    "cb_person_cred_hist_length", "loan_to_income"
]
categorical_features = [
    "person_home_ownership", "loan_intent", "cb_person_default_on_file"
]

# Section 6 creates loan_to_income before the split.
X = df[[
    "person_age", "person_income", "person_home_ownership", "person_emp_length",
    "loan_intent", "loan_amnt", "cb_person_default_on_file",
    "cb_person_cred_hist_length"
]].copy()

## Section 6: Feature Engineering

Create a loan-to-income ratio from the available loan amount and annual income. Missing or invalid values in either source remain missing and are handled by the numeric imputer. The dataset does not contain existing debt, utilization, payment history, late-payment counts, or loan duration, so those features are not invented.

In [ ]:
X["loan_to_income"] = X["loan_amnt"] / X["person_income"]
X["loan_to_income"] = X["loan_to_income"].replace([np.inf, -np.inf], np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training target proportions:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("proportion"))
print("Testing target proportions:")
display(y_test.value_counts(normalize=True).sort_index().to_frame("proportion"))

numeric_preprocessor = SimpleImputer(strategy="median")
categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

## Section 7: Random Forest Model

Train a Random Forest with 100 trees, a maximum depth of 10, and a fixed random seed for reproducibility. Balanced class weights help account for differences in class frequency.

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
print("Model training complete.")

## Section 8: Model Evaluation

Evaluate predictions on the held-out test data. Precision, recall, and F1 are reported with **bad credit (class 0)** as the positive class. ROC-AUC and the ROC curve also measure ranking of bad-credit cases.

In [ ]:
y_pred = model.predict(X_test)
class_labels = list(model.named_steps["classifier"].classes_)
bad_class_index = class_labels.index(0)
y_default_score = model.predict_proba(X_test)[:, bad_class_index]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (Bad Credit = 0)": precision_score(y_test, y_pred, pos_label=0, zero_division=0),
    "Recall (Bad Credit = 0)": recall_score(y_test, y_pred, pos_label=0, zero_division=0),
    "F1-Score (Bad Credit = 0)": f1_score(y_test, y_pred, pos_label=0, zero_division=0),
    "ROC-AUC (Bad Credit = 0)": roc_auc_score((y_test == 0).astype(int), y_default_score)
}

display(pd.Series(metrics, name="Test Set Result").to_frame())
print("Classification Report (0 = Bad Credit, 1 = Good Credit):")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Bad Credit / High Risk", "Good Credit / Low Risk"],
    zero_division=0
))

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1])

plt.figure(figsize=(6, 5))
sns.heatmap(
    conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Bad (0)", "Good (1)"],
    yticklabels=["Bad (0)", "Good (1)"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(
    (y_test == 0).astype(int), y_default_score,
    name="Bad Credit / Default"
)
plt.title("ROC Curve for Bad Credit")
plt.show()

feature_names = model.named_steps["preprocessor"].get_feature_names_out()
feature_importances = pd.Series(
    model.named_steps["classifier"].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(feature_importances.head(15).to_frame("importance"))
feature_importances.head(15).sort_values().plot(kind="barh", figsize=(9, 6), color="teal")
plt.xlabel("Feature Importance")
plt.title("Most Important Features")
plt.tight_layout()
plt.show()

## Section 9: Prediction

Predict the first five records from the test set. These are the same engineered columns used during training, and the fitted pipeline applies the identical imputation and encoding steps. The estimated default score is the model's score for class 0; it is not calibrated as a real-world probability.

In [ ]:
sample_X = X_test.head(5)
sample_actual = y_test.loc[sample_X.index]
sample_predicted = model.predict(sample_X)
sample_default_scores = model.predict_proba(sample_X)[:, bad_class_index]

predictions = pd.DataFrame({
    "actual_creditworthy": sample_actual,
    "predicted_creditworthy": sample_predicted,
    "estimated_default_score_uncalibrated": sample_default_scores
}, index=sample_X.index)

predictions["actual_label"] = predictions["actual_creditworthy"].map({1: "Good Credit", 0: "Bad Credit"})
predictions["predicted_label"] = predictions["predicted_creditworthy"].map({1: "Good Credit", 0: "Bad Credit"})
display(predictions)

## Section 10: Conclusion

The model classifies customers as good/low-risk (`1`) or bad/high-risk (`0`) using the available customer and loan fields. A Random Forest learns patterns from the training records, and the preprocessing pipeline ensures that training and prediction use the same transformations.

The results below are calculated from the actual held-out test set. Recall and precision for bad credit are especially relevant for default-risk screening. Whether performance is suitable depends on the measured results and the costs of missed defaults versus false alarms. Further validation on new data and calibration checks are needed before operational use.

In [ ]:
print("Held-out test results:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nConclusion:")
if metrics["ROC-AUC (Bad Credit = 0)"] >= 0.75:
    print("The model shows good ability to distinguish bad-credit cases on this test split.")
elif metrics["ROC-AUC (Bad Credit = 0)"] >= 0.60:
    print("The model shows some ability to distinguish bad-credit cases, but performance is moderate on this test split.")
else:
    print("The model's ability to distinguish bad-credit cases is limited on this test split.")
print("Review bad-credit precision and recall alongside the confusion matrix before deciding whether the model is suitable.")